<img src="../assets/header.gif" style="width:100%">
<hr style="color:#808080;">
<p align="center"><b>S H A G G Y&nbsp;&nbsp;&nbsp;—&nbsp;&nbsp;&nbsp;G R A D I E N T S</b></p>
<hr style="color:#808080;">

This notebook measures how much GPU memory a single gradient computation takes for a convolutional encoder, fed with random data that has the shape of the full domain. The same experiment is run for a 3D and a 2D encoder, each time with two losses, to check that the loss itself is not what fills the memory.

In [1]:
import math
import torch

from shaggy.loss import loss_geometry_embedding
from shaggy.models.cae import ConvEncoder

<hr style="color:#808080;">
<p align="center"><b>C O N F I G U R A T I O N</b></p>
<hr style="color:#808080;">

All the parameters that influence the notebook are grouped in the dictionary below. The number of input channels of each encoder is read from the shape of its data.

In [2]:
config = {
    "data": {
        "batch_size": 4,  # GME compares samples in pairs, it needs at least 2
        "shape_3d": (10, 48, 256, 568),  # (C, Z, Y, X)
        "shape_2d": (4, 256, 568),    # (C, Y, X)
    },
    "encoder_3d": {
        "spatial": 3,
        "out_channels": 128,
        "hid_channels": [16, 32, 64, 128],
        "hid_blocks": [3, 3, 3, 3],
        "hid_groups": [2, 2, 1, 1],
        "kernel_size": 3,
        "stride": 2,
        "pixel_shuffle": True,
        "ffn_factor": 1,
        "patch_size": 1,
        "periodic": False,
        "dropout": 0.02,
        "checkpointing": True,  # True trades compute for memory
        "identity_init": True,
    },
    "encoder_2d": {
        "spatial": 2,
        "out_channels": 128,
        "hid_channels": [16, 32, 64, 128],
        "hid_blocks": [3, 3, 3, 3],
        "hid_groups": [2, 2, 1, 1],
        "kernel_size": 3,
        "stride": 2,  # Each spatial size must be divisible by stride ** (depths - 1)
        "pixel_shuffle": True,
        "ffn_factor": 1,
        "patch_size": 1,
        "periodic": False,
        "dropout": 0.02,
        "checkpointing": True,  # True trades compute for memory
        "identity_init": True,
    },
}

device = torch.device("cuda")

<hr style="color:#808080;">
<p align="center"><b>T O O L S</b></p>
<hr style="color:#808080;">

Three small functions. The first counts the trainable parameters of a model. The second gives the shape of the latent code and the compression factor, i.e. how many times smaller the latent code is than the data. The third computes one gradient with each loss, the mean of the squared latent code and the GME, then prints the size of the gradient, the peak memory used by PyTorch and what is left on the GPU. If both peaks are close, the loss is not the problem.

In [3]:
def trainable_parameters(model: torch.nn.Module) -> int:
    r"""Counts the trainable parameters of a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def compression(model: ConvEncoder, shape: tuple) -> tuple:
    r"""Computes the latent shape and the compression factor for data of a given shape (C, L_1, ..., L_N)."""
    latent = (model.out_proj.out_channels, *(l // s for l, s in zip(shape[1:], model.scale)))
    return latent, math.prod(shape) // math.prod(latent)


LOSSES = {
    "Mean": lambda x, z: z.square().mean(),
    "GME": loss_geometry_embedding,
}


def gradient_memory(model: torch.nn.Module, x: torch.Tensor) -> None:
    r"""Computes one gradient per loss, then prints the memory it took."""
    for name, loss in LOSSES.items():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        z = model(x)
        loss(x, z).backward()

        gradient = sum(p.grad.numel() * p.grad.element_size() for p in model.parameters())
        peak = torch.cuda.max_memory_allocated()
        free, total = torch.cuda.mem_get_info()
        model.zero_grad(set_to_none=True)

        print(
            f"{name:4} | gradient: {gradient / 1e9:.3f} Go | peak: {peak / 1e9:.2f} Go "
            f"| free: {free / 1e9:.2f} / {total / 1e9:.2f} Go"
        )

<hr style="color:#808080;">
<p align="center"><b>3 D&nbsp;&nbsp;&nbsp;E N C O D E R</b></p>
<hr style="color:#808080;">

We build the 3D encoder, count its trainable parameters, then compute one gradient on a random tensor of shape (B, C, Z, Y, X). If the GPU runs out of memory, restart the kernel before trying again: the error keeps the tensors of the failed step alive.

In [4]:
shape = config["data"]["shape_3d"]
encoder = ConvEncoder(in_channels=shape[0], **config["encoder_3d"]).to(device)
x = torch.randn(config["data"]["batch_size"], *shape, device=device)

latent, factor = compression(encoder, shape)

print(f"Trainable parameters: {trainable_parameters(encoder):,}")
print(f"Compression         : x{factor} | latent shape: {latent}")

Trainable parameters: 7,713,824
Compression         : x40 | latent shape: (128, 6, 32, 71)


In [5]:
gradient_memory(encoder, x)

Mean | gradient: 0.031 Go | peak: 30.36 Go | free: 5.36 / 42.44 Go
GME  | gradient: 0.031 Go | peak: 30.36 Go | free: 3.86 / 42.44 Go


<hr style="color:#808080;">
<p align="center"><b>2 D&nbsp;&nbsp;&nbsp;E N C O D E R</b></p>
<hr style="color:#808080;">

Same experiment for the 2D encoder, on a random tensor of shape (B, C, Y, X). Reusing the names `encoder` and `x` frees the 3D experiment from the GPU.

In [ ]:
shape = config["data"]["shape_2d"]
encoder = ConvEncoder(in_channels=shape[0], **config["encoder_2d"]).to(device)
x = torch.randn(config["data"]["batch_size"], *shape, device=device)

latent, factor = compression(encoder, shape)

print(f"Trainable parameters: {trainable_parameters(encoder):,}")
print(f"Compression         : x{factor} | latent shape: {latent}")

In [ ]:
gradient_memory(encoder, x)